In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py as hf

import flax.linen as nn
import pickle
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib as mlp 
from matplotlib import cm

import seaborn as sns

# Distance from Source During Training

In [ ]:
def compute_distance(data: np.ndarray):
    """
    Compute the Euclidean distance from the center 
    for all time steps.
    """
    center = np.array([500.0, 500.0, 0.0])  # Fixed center
    distances = np.linalg.norm(data - center, axis=-1)
    
    return np.mean(distances, axis=-1), np.std(distances, axis=-1) / np.sqrt(data.shape[1])

In [ ]:
def compute_ensemble_average(temperature: str, indices: list):
    """
    Compute the ensemble average for one temperature
    """
    distance_array = []
    error_array = []
    
    # Collect the data
    for i in indices:
        with hf.File(f"{temperature}/{i}/deployment/trajectory.hdf5", "r") as db:
            data = db["colloids"]["Unwrapped_Positions"][:]
            
        distances, errors = compute_distance(data)
        distance_array.append(distances)
        error_array.append(errors)
        
    mean_distance = np.mean(distance_array, axis=0)
    experiment_error = np.sqrt(np.sum(np.array(error_array) ** 2, axis=0))
    ensemble_error = np.std(distance_array, axis=0) / np.sqrt(len(distance_array))
    
    return mean_distance, experiment_error, ensemble_error

In [ ]:
with hf.File("0K/9/deployment/trajectory.hdf5", "r") as db:
    low_positions = db["colloids"]["Unwrapped_Positions"][:]
    
with hf.File("350K/9/deployment/trajectory.hdf5", "r") as db:
    high_positions = db["colloids"]["Unwrapped_Positions"][:]

In [ ]:
results = {
    "0K": {"indices": [7, 9, 10]},
    "150K": {"indices": [1,2,3,4,6,7,8,9,10]},
    "273K": {"indices": [4,6,9,10]},
    "300K": {"indices": [3,5,6,7,9]},
    "350K": {"indices":  [2,3,5,6,9,10]}
}

for item in results:
    mean_distance, exp_err, ens_err = compute_ensemble_average(item, results[item]["indices"])
    
    results[item]["distances"] = mean_distance
    results[item]["exp_err"] = exp_err
    results[item]["ens_err"] = ens_err

# Policy Evaluation after Training

In [ ]:
class ActorNet(nn.Module):
    """A simple dense model."""

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=128)(x)
        x = nn.relu(x)
        x = nn.Dense(features=4)(x)
        return x

actor = ActorNet()

In [ ]:
path = "/auto.bee/work/stovey/work/PhD/microrobots-brownian/"
experiment = "find-center"
temperatures = [0, 150, 273, 300, 350]
ensembls = list(range(11))[1:]

ensembles = {0: [7, 9, 10], 150: [1,2,3,4,6,7,8,9,10], 273: [4,6,9,10], 300:  [3,5,6,7,9], 350:  [2,3,5,6,9,10]}
#ensembles = {0: [7], 150: [1], 273: [4], 300:  [3], 350:  [2]}

inputs = np.linspace(-1,1,10)


In [ ]:
temp_probabs = []
temp_error_probabs = []
for temp in temperatures:
    ensemble_porbs = []
    for ensemble in range(100):
        ensemble += 1 
        if ensemble in ensembles[temp]:
            outputs = []
            with open(path + f"{experiment}/{temp}K/{ensemble}/Models/ActorModel_0.pkl", "rb") as f:
                model_params, _, _, _ = pickle.load(f)

            for inputter in inputs:
                model_output = actor.apply({"params": model_params}, np.array([inputter]))
                outputs.append(model_output)
            outputs = np.array(outputs)
            wahrscheinlichkeiten = nn.softmax(outputs, axis=1)
            ensemble_porbs.append(wahrscheinlichkeiten)
        else:
            pass
    
    mean_ensemble_porbs = np.mean(np.array(ensemble_porbs),axis =0)
    error = np.mean((np.array(ensemble_porbs)-mean_ensemble_porbs)**2, axis=0)
    temp_error_probabs.append(error)
    temp_probabs.append(mean_ensemble_porbs)
probabilities_error = np.array(temp_error_probabs)
probabilities = np.array(temp_probabs)    

In [ ]:
fig, ax = plt.subplots(1, 4,figsize=(10, 3))
actions = ["CW", "Translate","CCW","Nothing"]
linestyles = [(0, (5, 5)), (0, (3, 5, 1, 5)), (0, (5, 1)), (0, (1, 1)), 'solid']
# colors = ['black'] * 5 #['blue', "red", 'green', 'black', "magenta"]

for i, temp in enumerate(temperatures): 
    for j in range(4):
        ax[j].plot(inputs, 
                   probabilities[i,:,j], 
                   linestyle=linestyles[i],
                   label=f"{temp}K",
                   color=colors[i])
        #ax[j].fill_between(inputs, 
        #               probabilities[i,:,j] - probabilities_error[i,:,j],
        #               probabilities[i,:,j] + probabilities_error[i,:,j] , 
         #              color=colors[i], 
        #               alpha=0.2)
        ax[j].set_ylim((0.0, 1.0))
        ax[j].set_title(actions[j])
    plt.legend()
    

# Plots

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(20, 15))

# Update the axes
left, bottom, width, height = [0.16, 0.73, 0.3, 0.14]
ax1 = ax[0, 0]
ax2 = fig.add_axes([left, bottom, width, height])
ax3 = ax[0, 1]

ax4 = ax[1, 0]
ax5 = ax[1, 1]

x = np.arange(results["150K"]["distances"].shape[0])

for k, item in enumerate(results):
    # Extract data
    distances = results[item]["distances"]
    exp_err = results[item]["exp_err"]
    ens_err = results[item]["ens_err"]
    
    # Full plot
    ax1.plot(distances,
             linestyle=linestyles[k]
            )
    ax1.fill_between(x, 
                     distances - exp_err, 
                     distances + exp_err, 
                     alpha=0.1
    )
    
    # Inset plot
    ax2.plot(distances,
             linestyle=linestyles[k])
    ax2.fill_between(x, 
                     distances - exp_err, 
                     distances + exp_err, 
                     alpha=0.3
    )
    # Distribution plot
    sns.kdeplot(distances[1000:], ax=ax3, label=item, fill=True)# linestyle=linestyles[k])
    
# Trajectory Plots
end = 300
r = 46 / 255
g = 41 / 255
b = 78 / 255

ax4.plot(low_positions[0, :, 0], low_positions[0, :, 1], "o", color="#D90368")
ax4.plot(low_positions[:end, :, 0], low_positions[:end, :, 1], color="#2E294E", alpha=0.3)
ax4.plot(low_positions[-1, :, 0], low_positions[-1, :, 1], "x", color="#FFD400")

ax5.plot(high_positions[0, :, 0], high_positions[0, :, 1], "o", color="#D90368")
ax5.plot(high_positions[:end, :, 0], high_positions[:end, :, 1], color="#2E294E", alpha=0.3)
ax5.plot(high_positions[-1, :, 0], high_positions[-1, :, 1], "x", color="#FFD400")
    
# Update axis 1 properties
ax1.set_xlabel("Time Step")
ax1.set_ylabel("Mean Distance to Source")
ax1.set_yscale("log")

# Update axis 2 properties
ax2.set_xlim(0, end)

# Update axis 3 properties
ax3.set_xlabel("Distances from Source")

# Update axis 4 properties
ax4.set_xlabel("x")
ax4.set_ylabel("y")

# Update axis 5 properties
ax5.set_xlabel("x")
ax5.set_ylabel("y")

# Set figure options and save
fig.legend(loc=(0.86, 0.7))
# plt.savefig("find-location-temp.png")


plt.show()

In [ ]:
actions = ["CW", "Translate","CCW","Nothing"]
linestyles = [(0, (5, 5)), (0, (3, 5, 1, 5)), (0, (5, 1)), (0, (1, 1)), 'solid']
colors =["#5c9ead","#e2c044","#326273","#2e933c","#e39774"]

In [ ]:
fig = plt.figure(figsize=(10, 10), layout="constrained")
spec = fig.add_gridspec(5, 4)

# policy evaluation (top row)
ax0 = fig.add_subplot(spec[0, 0])
ax1 = fig.add_subplot(spec[0, 1])
ax2 = fig.add_subplot(spec[0, 2])
ax3 = fig.add_subplot(spec[0, 3])
ax0.set_title('A)', loc='left', fontsize='medium')


# mean distances to the source (middles left but two)
ax4 = fig.add_subplot(spec[1, 0:2], label='B')
ax5 = fig.add_subplot(spec[2, 0:2])

# histo grams 
ax6 = fig.add_subplot(spec[1:3, 2:4])

# trajectory left
ax7 = fig.add_subplot(spec[3:6, 0:2])

# trajectory right 
ax8 = fig.add_subplot(spec[3:6, 2:4], label="E")

axs = [ax0, ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8]


for i, temp in enumerate(temperatures): 
    for j in range(4):
        axs[j].plot(inputs, 
                   probabilities[i,:,j], 
                   linestyle=linestyles[i],
                   label=f"{temp}K",
                   color=colors[i])
        #ax[j].fill_between(inputs, 
        #               probabilities[i,:,j] - probabilities_error[i,:,j],
        #               probabilities[i,:,j] + probabilities_error[i,:,j] , 
         #              color=colors[i], 
        #               alpha=0.2)
        axs[j].set_ylim((-0.1, 1.1))
        axs[j].set_title(actions[j])
        axs[j].set_xlabel("Concentration Change")
        axs[j].set_ylabel("Probability")
        
x = np.arange(results["150K"]["distances"].shape[0])

for k, item in enumerate(results):
    # Extract data
    distances = results[item]["distances"]
    exp_err = results[item]["exp_err"]
    ens_err = results[item]["ens_err"]
    #  Distance Plot full
    ax4.plot(distances,
             linestyle=linestyles[k],
             color=colors[k]
            )
    ax4.fill_between(x, 
                     distances - exp_err, 
                     distances + exp_err, 
                     alpha=0.1,
                     color=colors[k]
    )
    
    # Distance Plot beginning 
    ax5.plot(distances,
             linestyle=linestyles[k],
             label=f"{item}",
             color=colors[k]
            )
    ax5.fill_between(x, 
                     distances - exp_err, 
                     distances + exp_err, 
                     alpha=0.3,
                     color=colors[k]
    )
    
    # Distribution plot
    sns.kdeplot(distances[1000:],
                ax=ax6, 
                label=item, 
                fill=True,
                color=colors[k]
               )

    
# Trajectory Plots
end = 250

# Low Temperature 
ax7.plot(low_positions[:end, :, 0], 
         low_positions[:end, :, 1], 
         color=colors[0], 
         alpha=0.8,
        )
ax7.plot(low_positions[0, :, 0], 
         low_positions[0, :, 1], 
         "d", 
         color=colors[3],
         label="starting point"
        )
ax7.plot(low_positions[end, :, 0], 
         low_positions[end, :, 1], 
         "2", 
         color='black',
         label="end point"
        )

# High Temperature 
ax8.plot(high_positions[:end, :, 0], 
         high_positions[:end, :, 1], 
         color=colors[0], 
         alpha=0.8,
        )
ax8.plot(high_positions[0, :, 0], 
         high_positions[0, :, 1], 
         "d", 
         color=colors[3],
         label="starting point")
ax8.plot(high_positions[end, :, 0], 
         high_positions[end, :, 1], 
         "2", 
         color='black',
         label="end point")

# Update axis 3 
ax3.legend()

# Update axis 4 properties
ax4.set_xlabel("Time Step")
ax4.set_ylabel("Mean Distance to Source")
ax4.set_title('B)', loc='left', fontsize='medium')
ax4.set_yscale("log")

# Update axis 5 properties
ax5.set_xlim(0, end)
ax5.set_xlabel("Time Step")
ax5.set_ylabel("Mean Distance to Source")
ax5.set_title('C)', loc='left', fontsize='medium')
ax5.legend()

# Update axis 6 properties
ax6.set_xlabel("Distances from Source")
ax6.set_title('D)', loc='left', fontsize='medium')
ax6.legend()

# Update axis 7 properties
ax7.set_xlabel("x")
ax7.set_ylabel("y")
ax7.set_title('E)', loc='left', fontsize='medium')
ax7.legend()

# Update axis 8 properties
ax8.set_xlabel("x")
ax8.set_ylabel("y")
ax8.set_title('F)', loc='left', fontsize='medium')
ax8.legend()

# Set figure options and save

plt.savefig("find-location-temp.png")


In [ ]:
# Make data
def potential(x,y, x_0 = 0.5, y_0 = 0.5):
    r = np.sqrt((x-x_0)**2 + (y-y_0)**2)
    return 1 / r


# Surface
x_in = np.linspace(-1, 2, 100)
y_in = np.linspace(-1, 2, 100)

xs, ys = np.meshgrid(x_in, y_in)

z = potential(xs, ys)

# Point 
x_p = np.array([1.3, 1.0])
y_p = np.array([1.3, 1.0])

# trajecotry 
ts = np.linspace(0,1, 25)
start = np.array([1.3, 1.3])
end = np.array([1.0, 1.0])
dist = end - start
dist = dist 
pos = [start]
for t in ts: 
    pos.append(start+t*dist)
pos = np.array(pos).T
x_i = pos[0]
y_i = pos[1]

In [ ]:
import plotly.graph_objects as go
import pandas as pd

colors_mate =["#5c9ead","#e2c044","#326273","#2e933c","#e39774"]

fig = go.Figure(data=go.Surface(x=xs, y=ys, z=z, 
                                surfacecolor= 0.5 * np.ones_like(z), 
                                showscale=True,
                                colorscale=colors,
                                   contours = {
        "z": {"show": True, "start": 0.0, "end": None, "size": 0.01}
    },)
                               )

fig.update_layout(
    title='Mt Bruno Elevation',
    width=800, height=800,
    margin=dict(t=40, r=0, l=20, b=20),
)

fig.update_layout(scene = dict(
                    xaxis = dict(
                        range=[-1, 2],    
                         showbackground=False,),
                    yaxis = dict(
                        range=[-1, 2],
                        showbackground=False,),
                    zaxis = dict(
                        range=[0,5],
                        showbackground=False,),),
                    width=700,
                  )

name = 'default'
# Default parameters which are used when `layout.scene.camera` is not provided
camera = dict(
    up=dict(x=0, y=0, z=1),
    center=dict(x=0, y=0, z=0),
    eye=dict(x=1.25, y=1.25, z=1.25)
)

fig.update_layout(scene_camera=camera, title=name)


In [ ]:

import plotly.graph_objs as go

import numpy as np

plot_objects = []

surface = go.Surface(x=xs, y=ys, z=z,
           surfacecolor= 0.5 * np.ones_like(z), 
           showscale=False,
           colorscale=colors,
          )
plot_objects.append(surface)

points = go.Scatter3d(x=x_p, y=y_p, z=potential(x_p,y_p)+np.array([0.12]), 
                      mode="markers",
                      marker=dict(size=10, 
                                  color="#326273",
                                  line=dict(width=2,
                                            color="#5c9ead"),
                                 ),
                     )


plot_objects.append(points)



# Creating the plot

line_marker = dict(color="#e2c044", width=2)
trajectory_marker = dict(color="#e2c044", width=9)
for i, j, k in zip(xs, ys, z):
    plot_objects.append(go.Scatter3d(x=i, y=j, z=k, mode='lines', line=line_marker))
    plot_objects.append(go.Scatter3d(x=j, y=i, z=k, mode='lines', line=line_marker))

plot_objects.append(go.Scatter3d(x=x_i, y=y_i, z=potential(x_i,y_i),mode="lines", line=trajectory_marker))
    
layout = go.Layout(
    title='Wireframe Plot',
    scene=dict(
        xaxis=dict(showbackground=False),
        yaxis=dict(showbackground=False),
        zaxis=dict(showbackground=False),
    ),
    showlegend=False,
)

fig = go.Figure(data=plot_objects, layout=layout)
fig.update_layout(
    title='Concentration Field',
    width=800, height=800,
    margin=dict(t=40, r=0, l=20, b=20),
    scene = dict(xaxis = dict(range=[-1.5, 2.5], showbackground=False,),
                 yaxis = dict(range=[-1.5, 2.5], showbackground=False,),
                 zaxis = dict(range=[0,2], showbackground=False,),),
)


fig.show()